# OpticksHits: hits per sensor + sensor positions from GDML

Reads the `OpticksHits` TTree written by OpticalSims (`gdml_det`) with **uproot**, counts hits per
`SensorID`, and draws them as a heat map on the detector's sensors, whose positions are read directly
from the **GDML** file (no Geant4/ROOT needed).

| Function | What it does |
|---|---|
| `read_gdml_sensors(gdml)` | Walks the GDML volume tree and returns every sensitive placement (`SensDet`/`PD` = `PhotonDetector`) with its copy number and global position |
| `draw_sensor_positions(sensors)` | 3D view + unrolled (barrel φ–z, caps x–y) view of all sensors |
| `load_hits(root_file)` | `OpticksHits` → pandas DataFrame (optionally one event / a list of events) |
| `hits_per_sensor(hits, sensors)` | Hit count for every sensor (sensors with 0 hits included) |
| `plot_hit_heatmap(sensors, counts)` | Heat map on the unrolled cylinder (`layout="unrolled"`) or in 3D (`layout="3d"`) |

### Running the notebook
Set `GDML_FILE` and `ROOT_FILE` below to **absolute paths** and **Run → Run All Cells**.
Head-less alternative (writes an executed copy with all figures)


Use **absolute paths** (`~` and `$VARS` are expanded). Relative paths would be resolved from the
directory Jupyter was started in, not from where the files are. Find a file's absolute path with
`realpath test.root` in a terminal.


## Configuration



In [ ]:
from pathlib import Path

GDML_FILE = "OpticalSims/GDML/SK-like_opticks.gdml"   # geometry used for the run
ROOT_FILE = "opticks_test/test_SK.root"               # gdml_det output (/RunAction/output/file)
TREE_NAME = "OpticksHits"   # or "Geant4Hits" (CPU hits: same evtID/SensorID/x/y/z/t/wavelength columns)
INLINE_FORMAT = "jpeg"      # "jpeg" (small), "png" (sharper, ~5x bigger) or None (show nothing inline; use SAVE_DIR)
INLINE_DPI    = 72          # resolution of the embedded figures
EVENT_DISPLAYS = 1         # how many single-event heat maps to draw in section 4
EVENTS    = None      # None = all events, an int = one event, a list = several events
LOG_COLOR = True      # log colour scale for the heat map
SAVE_DIR  = "/plots"      # e.g. "plots" to also save PNGs

In [ ]:
import os


def _find_by_name(name, roots, max_depth=5, limit=10):
    # Look for files called `name` under `roots` (bounded depth) to suggest the right path.
    hits = []
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if name in filenames:
                hits.append(Path(dirpath) / name)
                if len(hits) >= limit:
                    return hits
    return hits


def absolute_path(p, what="file", must_exist=True):
    # Expand ~ / $VARS, require an absolute path, and (optionally) check that it exists.
    p = Path(os.path.expandvars(os.path.expanduser(str(p))))
    if not p.is_absolute():
        print(f"WARNING: {what} path {str(p)!r} is relative; resolving it from {Path.cwd()}")
        p = Path.cwd() / p
    p = p.resolve()
    if must_exist and not p.exists():
        found = list(dict.fromkeys(_find_by_name(p.name, [Path.home(), p.parent.parent])))  # de-duplicated
        hint = ("\n  files with that name:\n    " + "\n    ".join(map(str, found))) if found else \
               f"\n  no file called {p.name!r} found under {Path.home()}"
        raise FileNotFoundError(f"{what} not found: {p}{hint}")
    return p


GDML_FILE = absolute_path(GDML_FILE, "GDML file")
ROOT_FILE = absolute_path(ROOT_FILE, "ROOT file")
SAVE_DIR = absolute_path(SAVE_DIR, "SAVE_DIR", must_exist=False) if SAVE_DIR else None
print("GDML_FILE =", GDML_FILE)
print("ROOT_FILE =", ROOT_FILE)
if SAVE_DIR:
    print("SAVE_DIR  =", SAVE_DIR)

## Imports

In [ ]:
import math
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import Circle
import uproot

pd.set_option("display.width", 140)
print("uproot", uproot.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

plt.rcParams["figure.dpi"] = INLINE_DPI
if INLINE_FORMAT == "jpeg":
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 70, "optimize": True})
elif INLINE_FORMAT == "png":
    set_matplotlib_formats("png")
else:                                   # nothing embedded: figures only go to SAVE_DIR
    plt.ioff()
    set_matplotlib_formats("png")
print(f"inline figures: {INLINE_FORMAT or 'off'} @ {INLINE_DPI} dpi")

## Sensor positions from the GDML file

Global transforms follow Geant4's GDML reader exactly: a physvol `<rotation x y z>` builds
`M = Rz·Ry·Rx` and the placement uses `M⁻¹`, i.e. `p_global = t + M⁻¹ p_local`, composed down the tree.
Supports inline or referenced (`positionref`/`rotationref`) placements, `unit`/`aunit` attributes,
`<constant>`/`<variable>`/`<quantity>` expressions, and any nesting depth.

In [ ]:
_UNITS = {"mm": 1.0, "cm": 10.0, "m": 1000.0, "km": 1e6, "um": 1e-3, "nm": 1e-6,
          "rad": 1.0, "mrad": 1e-3, "deg": math.pi / 180.0, "degree": math.pi / 180.0}
_ANG_DEFAULT, _LEN_DEFAULT = "rad", "mm"     # GDML defaults


class _Expr:
    # Evaluates GDML numeric attributes ("16900", "2*R", "180*deg", "pi/2") with <define> constants.
    def __init__(self):
        self.ns = {"pi": math.pi, "e": math.e, "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
                   "tan": math.tan, "asin": math.asin, "acos": math.acos, "atan": math.atan,
                   "atan2": math.atan2, "exp": math.exp, "log": math.log, "abs": abs, "pow": pow}
        self.ns.update(_UNITS)

    def __call__(self, s, default=0.0):
        if s is None:
            return default
        try:
            return float(s)
        except ValueError:
            return float(eval(str(s).replace("^", "**"), {"__builtins__": {}}, self.ns))

    def define(self, name, value):
        self.ns[name] = self(value)


def _rot_matrix(ax, ay, az):
    # CLHEP: rot.rotateX(ax); rot.rotateY(ay); rot.rotateZ(az)  ->  M = Rz @ Ry @ Rx
    cx, sx, cy, sy, cz, sz = math.cos(ax), math.sin(ax), math.cos(ay), math.sin(ay), math.cos(az), math.sin(az)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def read_gdml_sensors(gdml_path, sensor_aux=("SensDet", "PD"), sensor_value="PhotonDetector",
                      sensor_lvs=None, verbose=True):
    # Return a DataFrame with one row per sensitive placement:
    #   copynumber, lv, pv, x, y, z [mm, sensor-volume origin], dir_x/y/z (sensor local +x in global frame),
    #   depth, surface ('barrel' / 'top' / 'bottom', for cylinder-like detectors)
    # Sensors = LVs carrying <auxiliary auxtype in sensor_aux auxvalue=sensor_value/> (as OpticalSims'
    # DetectorConstruction does), or the explicit list `sensor_lvs`.
    root = ET.parse(str(gdml_path)).getroot()
    for el in root.iter():                          # drop any XML namespace prefixes
        if isinstance(el.tag, str) and "}" in el.tag:
            el.tag = el.tag.split("}", 1)[1]
    ev = _Expr()

    positions, rotations = {}, {}
    for d in root.findall("define"):
        for el in d:
            name = el.get("name")
            if el.tag in ("constant", "variable"):
                ev.define(name, el.get("value"))
            elif el.tag == "quantity":
                ev.define(name, f"({el.get('value')})*{el.get('unit', '1')}")
    for d in root.findall("define"):
        for el in d:
            if el.tag == "position":
                positions[el.get("name")] = el
            elif el.tag == "rotation":
                rotations[el.get("name")] = el

    def vec(el, kind):
        if el is None:
            return np.zeros(3)
        u = _UNITS.get(el.get("unit", _LEN_DEFAULT if kind == "pos" else _ANG_DEFAULT), 1.0)
        return np.array([ev(el.get(k), 0.0) for k in "xyz"]) * u

    structure = root.find("structure")
    volumes = {v.get("name"): v for v in structure if v.tag in ("volume", "assembly")}

    sensors = set(sensor_lvs or [])
    if not sensor_lvs:
        for name, v in volumes.items():
            for aux in v.findall("auxiliary"):
                if aux.get("auxtype") in sensor_aux and aux.get("auxvalue") == sensor_value:
                    sensors.add(name)
    if not sensors:
        raise RuntimeError("no sensitive volumes found (no SensDet/PD = PhotonDetector auxiliary)")

    world = root.find("setup/world").get("ref")
    rows, stack = [], [(world, np.zeros(3), np.eye(3), 0)]
    while stack:
        lv, t, R, depth = stack.pop()
        for pv in volumes[lv].findall("physvol"):
            ref = pv.find("volumeref")
            if ref is None:
                raise NotImplementedError("physvol without volumeref (inline <file>) is not supported")
            child = ref.get("ref")
            pos = pv.find("position")
            if pos is None and pv.find("positionref") is not None:
                pos = positions[pv.find("positionref").get("ref")]
            rot = pv.find("rotation")
            if rot is None and pv.find("rotationref") is not None:
                rot = rotations[pv.find("rotationref").get("ref")]
            Rloc = _rot_matrix(*vec(rot, "rot")).T          # GDML -> Geant4 uses the inverse
            tg, Rg = t + R @ vec(pos, "pos"), R @ Rloc
            if child in sensors:
                d = Rg[:, 0]
                rows.append((int(pv.get("copynumber", 0)), child, pv.get("name", child + "_PV"),
                             *tg, *d, depth + 1))
            elif child in volumes:
                stack.append((child, tg, Rg, depth + 1))

    df = pd.DataFrame(rows, columns=["copynumber", "lv", "pv", "x", "y", "z",
                                     "dir_x", "dir_y", "dir_z", "depth"])
    df = df.sort_values("copynumber", kind="stable").reset_index(drop=True)

    # cylinder bookkeeping: caps are the sensors sitting at the extreme z planes
    tol = 1.0  # mm
    zmax, zmin = df.z.max(), df.z.min()
    df["surface"] = np.where(df.z > zmax - tol, "top", np.where(df.z < zmin + tol, "bottom", "barrel"))
    df.attrs["sensor_radius"] = _sensor_radius(root, volumes, sorted(sensors), ev)
    df.attrs["gdml"] = str(gdml_path)

    if verbose:
        print(f"{gdml_path}: {len(df)} sensors in LVs {sorted(sensors)}")
        print(df.groupby(["lv", "surface"]).size().rename("n").to_string())
        dup = df.copynumber.duplicated().sum()
        if dup:
            print(f"WARNING: {dup} duplicated copy numbers -> SensorID is not unique per sensor")
        print(f"sensor radius ~ {df.attrs['sensor_radius']} mm")
    return df


def _sensor_radius(root, volumes, sensor_lvs, ev):
    # Radius of the first <orb>/<sphere> found in a sensor LV's solid tree (None if there is none).
    solids = {s.get("name"): s for s in root.find("solids")}
    for lv in sensor_lvs:
        todo = [volumes[lv].find("solidref").get("ref")]
        while todo:
            s = solids.get(todo.pop())
            if s is None:
                continue
            u = _UNITS.get(s.get("lunit", "mm"), 1.0)
            if s.tag == "orb":
                return ev(s.get("r")) * u
            if s.tag == "sphere":
                return ev(s.get("rmax")) * u
            todo += [c.get("ref") for c in s if c.tag in ("first", "second")]
    return None

## Unrolled cylinder layout

Barrel: arc length `R·φ` (horizontal) vs `z`; top cap drawn above, bottom cap below, each rotated so that
φ = 0 touches the middle of the barrel — the usual Super-K event-display layout.

In [ ]:
def unrolled_xy(sensors):
    # 2D coordinates (mm) of every sensor in the unrolled-cylinder layout.
    x, y, z, s = (sensors[k].to_numpy() for k in ("x", "y", "z", "surface"))
    rb = np.hypot(x, y)
    R = rb[s == "barrel"].max() if np.any(s == "barrel") else rb.max()
    ztop = z[s == "top"].max() if np.any(s == "top") else z.max()
    zbot = z[s == "bottom"].min() if np.any(s == "bottom") else z.min()
    u, v = np.empty_like(x), np.empty_like(x)
    b, t, bo = s == "barrel", s == "top", s == "bottom"
    u[b], v[b] = R * np.arctan2(y[b], x[b]), z[b]
    u[t], v[t] = y[t], ztop + R - x[t]          # phi=0 at the bottom of the top disc
    u[bo], v[bo] = y[bo], zbot - R + x[bo]      # phi=0 at the top of the bottom disc
    return u, v, dict(R=R, ztop=ztop, zbot=zbot)


def _outline(ax, geo, **kw):
    R, zt, zb = geo["R"], geo["ztop"], geo["zbot"]
    kw = {"color": "0.6", "lw": 0.8, **kw}
    ax.plot([-math.pi * R, math.pi * R, math.pi * R, -math.pi * R, -math.pi * R], [zb, zb, zt, zt, zb], **kw)
    for zc in (zt + R, zb - R):
        ax.add_patch(Circle((0, zc), R, fill=False, **kw))


def _save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(SAVE_DIR) / f"{name}.png", dpi=150, bbox_inches="tight")
    if not INLINE_FORMAT:
        plt.close(fig)

## Draw every sensor position (GDML)

In [ ]:
def draw_sensor_positions(sensors, title=None, color_by="lv", show_ids=False, figsize=(15, 7)):
    # 3D scatter of all sensors + the unrolled layout. color_by: 'lv' or 'surface'.
    title = title or f"{len(sensors)} sensors in {Path(sensors.attrs.get('gdml', '')).name}"
    groups = sensors.groupby(color_by)
    fig = plt.figure(figsize=figsize)
    ax3 = fig.add_subplot(1, 2, 1, projection="3d")
    for (name, g), c in zip(groups, plt.rcParams["axes.prop_cycle"].by_key()["color"]):
        ax3.scatter(g.x / 1e3, g.y / 1e3, g.z / 1e3, s=0.5, color=c, label=f"{name} ({len(g)})")
    ext = np.ptp(sensors[["x", "y", "z"]].to_numpy(), axis=0)
    ax3.set_box_aspect(ext / ext.max())
    ax3.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title=title)
    ax3.legend(markerscale=10, loc="upper left", fontsize=8)

    ax = fig.add_subplot(1, 2, 2)
    u, v, geo = unrolled_xy(sensors)
    for (name, g), c in zip(groups, plt.rcParams["axes.prop_cycle"].by_key()["color"]):
        m = sensors.index.isin(g.index)
        ax.scatter(u[m] / 1e3, v[m] / 1e3, s=0.6, color=c, label=name)
    if show_ids:
        for i in range(0, len(sensors), max(1, len(sensors) // 60)):
            ax.annotate(str(sensors.copynumber.iat[i]), (u[i] / 1e3, v[i] / 1e3), fontsize=5)
    _outline(ax, {k: val / 1e3 for k, val in geo.items()})
    ax.set_aspect("equal")
    ax.set(xlabel=r"barrel: $R\,\phi$ [m]   |   caps: y [m]", ylabel="z [m]", title="unrolled cylinder")
    fig.tight_layout()
    _save(fig, "sensor_positions")
    return fig

## Read the `OpticksHits` tree

In [ ]:
HIT_BRANCHES = ["evtID", "hit_Id", "SensorID", "x", "y", "z", "t", "wavelength", "boundary"]


def load_hits(root_file, tree=TREE_NAME, events=None, branches=None):
    # OpticksHits -> pandas DataFrame. events: None (all), int, or list of evtIDs.
    with uproot.open(root_file) as f:
        if tree not in f:
            raise KeyError(f"{tree!r} not in {root_file}; keys: {f.keys()}")
        t = f[tree]
        have = set(t.keys())
        want = [b for b in (branches or HIT_BRANCHES) if b in have]
        missing = [b for b in (branches or HIT_BRANCHES) if b not in have]
        if missing:
            print("note: branches not in file:", missing)
        df = t.arrays(want, library="pd")
        if events is not None:
            ev = [events] if np.isscalar(events) else list(events)
            df = df[df.evtID.isin(ev)]
    df = df.reset_index(drop=True)
    print(f"{root_file}:{tree}: {len(df):,} hits, events {[int(e) for e in sorted(df.evtID.unique())[:10]]}"
          f"{' ...' if df.evtID.nunique() > 10 else ''} ({df.evtID.nunique()} events)")
    return df


def hits_per_sensor(hits, sensors, per_event=False):
    # Hit count for every sensor in `sensors` (ordered like sensors), zero-hit sensors included.
    counts = hits.SensorID.value_counts()
    unknown = counts.index.difference(sensors.copynumber)
    if len(unknown):
        print(f"WARNING: {counts[unknown].sum()} hits on {len(unknown)} SensorIDs not in the GDML "
              f"(e.g. {list(unknown[:5])}) - wrong GDML file?")
    n = sensors.copynumber.map(counts).fillna(0).to_numpy()
    if per_event:
        n = n / max(hits.evtID.nunique(), 1)
    return n

## Heat map of hits per sensor

In [ ]:
def plot_hit_heatmap(sensors, counts, layout="unrolled", log=LOG_COLOR, title="Hits per sensor",
                     cmap="viridis", zero_color="0.88", vmax=None, label="hits", figsize=(10, 11), ax=None,
                     save_as=None):
    # layout='unrolled': every sensor drawn as a disc of its real radius on the unrolled cylinder.
    # layout='3d'      : 3D scatter coloured by counts.
    counts = np.asarray(counts, float)
    hit = counts > 0
    vmax = vmax or (counts.max() if hit.any() else 1)
    norm = LogNorm(vmin=max(counts[hit].min(), 1e-3) if hit.any() else 1, vmax=vmax) if log \
        else Normalize(0, vmax)

    if layout == "3d":
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(projection="3d")
        ax.scatter(sensors.x[~hit] / 1e3, sensors.y[~hit] / 1e3, sensors.z[~hit] / 1e3, s=0.3, color=zero_color)
        sc = ax.scatter(sensors.x[hit] / 1e3, sensors.y[hit] / 1e3, sensors.z[hit] / 1e3,
                        c=counts[hit], s=3, cmap=cmap, norm=norm)
        ext = np.ptp(sensors[["x", "y", "z"]].to_numpy(), axis=0)
        ax.set_box_aspect(ext / ext.max())
        ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]", title=title)
        fig.colorbar(sc, ax=ax, shrink=0.6, label=label)
        _save(fig, save_as or "hit_heatmap_3d")
        return fig

    u, v, geo = unrolled_xy(sensors)
    r = (sensors.attrs.get("sensor_radius") or 250.0) / 1e3
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
    order = np.argsort(counts)                         # brightest drawn last
    blank = PatchCollection([Circle((u[i] / 1e3, v[i] / 1e3), r) for i in order if not hit[i]],
                            facecolor=zero_color, edgecolor="none")
    lit = PatchCollection([Circle((u[i] / 1e3, v[i] / 1e3), r) for i in order if hit[i]],
                          cmap=cmap, norm=norm, edgecolor="none")
    lit.set_array(counts[order][hit[order]])
    ax.add_collection(blank)
    ax.add_collection(lit)
    _outline(ax, {k: val / 1e3 for k, val in geo.items()})
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.set(xlabel=r"barrel: $R\,\phi$ [m]   |   caps: y [m]", ylabel="z [m]",
           title=f"{title}  ({int(hit.sum())}/{len(counts)} sensors hit, {counts.sum():,.0f} {label})")
    fig.colorbar(lit, ax=ax, shrink=0.7, label=label)
    _save(fig, save_as or "hit_heatmap_unrolled")
    return fig


def check_hits_vs_sensors(hits, sensors, bins=100):
    # Distance between each hit and the origin of its SensorID volume; for the LUCiD-style PMTs
    # (sphere centred on the wall) every hit should sit at ~ the sensor radius.
    pos = sensors.set_index("copynumber")[["x", "y", "z"]]
    ok = hits.SensorID.isin(pos.index)
    d = np.linalg.norm(hits.loc[ok, ["x", "y", "z"]].to_numpy() - pos.loc[hits.SensorID[ok]].to_numpy(), axis=1)
    fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
    axs[0].hist(d, bins=bins)
    axs[0].set(xlabel="|hit - sensor centre| [mm]", ylabel="hits", title="hit-to-sensor distance")
    if sensors.attrs.get("sensor_radius"):
        axs[0].axvline(sensors.attrs["sensor_radius"], color="C3", ls="--", label="sensor radius")
        axs[0].legend()
    if "t" in hits:
        axs[1].hist(hits.t, bins=bins)
        axs[1].set(xlabel="t [ns]", ylabel="hits", title="hit time")
    if "wavelength" in hits:
        axs[2].hist(hits.wavelength, bins=bins)
        axs[2].set(xlabel="wavelength [nm]", ylabel="hits", title="hit wavelength")
    fig.tight_layout()
    _save(fig, "hit_checks")
    print(f"hit-to-sensor distance: median {np.median(d):.1f} mm, 99% < {np.percentile(d, 99):.1f} mm; "
          f"{(~ok).sum()} hits with unknown SensorID")
    return fig

## 1. Sensor positions

In [ ]:
sensors = read_gdml_sensors(GDML_FILE)
sensors.head()

In [ ]:
draw_sensor_positions(sensors);

## 2. Hits

In [ ]:
hits = load_hits(ROOT_FILE, events=EVENTS)
hits.head()

In [ ]:
print(hits.groupby("evtID").size().rename("hits per event").describe().to_string())

## 3. Heat map — all selected events

In [ ]:
counts = hits_per_sensor(hits, sensors)
plot_hit_heatmap(sensors, counts, title=f"{ROOT_FILE.name}: hits per sensor");

In [ ]:
plot_hit_heatmap(sensors, counts, layout="3d", title="hits per sensor (3D)", figsize=(9, 8));


## 4. One event at a time

In [ ]:
for evt in sorted(hits.evtID.unique())[:10]:
    c = hits_per_sensor(hits[hits.evtID == evt], sensors)
    plot_hit_heatmap(sensors, c, title=f"event {evt}", figsize=(8, 9), save_as=f"hit_heatmap_evt{evt}")

In [ ]:
check_hits_vs_sensors(hits, sensors);

In [ ]:
# Top-10 sensors by number of hits
top = sensors.assign(hits=counts).nlargest(10, "hits")[["copynumber", "lv", "surface", "x", "y", "z", "hits"]]
top